In [ ]:
import requests

# 1. Show the raw API response structure (JSON)
url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
params = {
    'parameters': 'T2M,RH2M,PRECTOTCORR,WS10M',
    'community': 'RE',
    'longitude': 74.3587,
    'latitude': 31.5204,
    'start': '20230101',
    'end': '20230107',
    'format': 'JSON'
}
response = requests.get(url, params=params, timeout=30)
data = response.json()

print("Raw JSON keys:", list(data.keys()))
print("\nProperties keys:", list(data['properties'].keys()))
print("\nSample of T2M data:")
print(data['properties']['parameter']['T2M'])

Raw JSON keys: ['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times']

Properties keys: ['parameter']

Sample of T2M data:
{'20230101': 11.95, '20230102': 12.25, '20230103': 11.41, '20230104': 10.18, '20230105': 10.64, '20230106': 10.92, '20230107': 12.08}


In [ ]:
import pandas as pd
import numpy as np

# 2. The daily DataFrame (.head())
params_data = data['properties']['parameter']
df_daily = pd.DataFrame(params_data)

# IMMEDIATELY replace -999.0 with np.nan
df_daily = df_daily.replace(-999.0, np.nan)

df_daily.index = pd.to_datetime(df_daily.index, format='%Y%m%d')
df_daily.index.name = 'date'
df_daily.head()

,T2M,RH2M,PRECTOTCORR,WS10M
date,,,,
2023-01-01,11.95,51.44,0.0,1.64
2023-01-02,12.25,40.31,0.0,1.48
2023-01-03,11.41,39.18,0.0,2.17
2023-01-04,10.18,43.61,0.0,2.52
2023-01-05,10.64,37.98,0.0,2.31


In [ ]:
import sys
sys.path.append('..')
from backend.utils.nasa_fetcher import fetch_weather
import pandas as pd

# Punjab districts only — 36 coordinates
district_coords = {
    "Lahore":          (31.5204, 74.3587),
    "Faisalabad":      (31.4504, 73.1350),
    "Rawalpindi":      (33.5651, 73.0169),
    "Gujranwala":      (32.1877, 74.1945),
    "Multan":          (30.1575, 71.5249),
    "Sheikhupura":     (31.7167, 73.9850),
    "Kasur":           (31.1167, 74.4500),
    "Sialkot":         (32.4945, 74.5229),
    "Sargodha":        (32.0836, 72.6711),
    "Bahawalpur":      (29.3956, 71.6836),
    "Jhang":           (31.2681, 72.3181),
    "Gujrat":          (32.5736, 74.0790),
    "Rahim Yar Khan":  (28.4202, 70.2952),
    "Sahiwal":         (30.6706, 73.1064),
    "Okara":           (30.8138, 73.4534),
    "Narowal":         (32.1000, 74.8700),
    "Hafizabad":       (32.0714, 73.6883),
    "Mandi Bahauddin": (32.5864, 73.4917),
    "Pakpattan":       (30.3436, 73.3869),
    "Chiniot":         (31.7200, 72.9800),
    "Khushab":         (32.2969, 72.3527),
    "Bhakkar":         (31.6278, 71.0642),
    "Layyah":          (30.9614, 70.9394),
    "Muzaffargarh":    (30.0736, 71.1936),
    "Lodhran":         (29.5333, 71.6333),
    "Vehari":          (30.0453, 72.3519),
    "Khanewal":        (30.3017, 71.9322),
    "Toba Tek Singh":  (30.9700, 72.4800),
    "Nankana Sahib":   (31.4500, 73.7100),
    "Attock":          (33.7664, 72.3601),
    "Chakwal":         (32.9328, 72.8571),
    "Jhelum":          (32.9350, 73.7203),
    "Mianwali":        (32.5836, 71.5433),
    "D.G. Khan":       (30.0500, 70.6333),
    "Rajanpur":        (29.1042, 70.3300),
    "Bahawalnagar":    (29.9978, 73.2536),
}

# fetch NASA weather per district — 36 API calls, takes ~8 minutes
weather_frames = []
failed = []

for district, (lat, lon) in district_coords.items():
    print(f"Fetching {district}...")
    try:
        df_wx = fetch_weather(lat, lon, "20160101", "20201231")
        df_wx["district"] = district
        weather_frames.append(df_wx)
    except Exception as e:
        print(f"  FAILED {district}: {e}")
        failed.append(district)

if failed:
    print(f"\nFailed districts — re-run manually: {failed}")
else:
    print("\nAll 36 districts fetched successfully")

df_weather = pd.concat(weather_frames, ignore_index=True)
print(f"Weather shape: {df_weather.shape}")

Fetching Punjab...
Fetching Sindh...
Fetching KPK...
Fetching Balochistan...
Fetching ICT...
Fetching AJK...
Fetching GB...
Merged shape: (1680, 11)
Saved.


In [24]:
import pandas as pd


def convertToCSV(file, output_file=None, **kwargs):
    df = pd.read_excel(file, **kwargs)
    
    if output_file is None:
        output_file = file.rsplit('.', 1)[0] + '.csv'
    
    df.to_csv(
        output_file,
        index=False,
        encoding='utf-8-sig'
    )
    
    print(f"Converted: {file} → {output_file}")
    return output_file




In [46]:
import pandas as pd
import numpy as np


# Realistic week-of-month patterns for off-season months
# Even "normal" months have slight variation due to reporting delays, 
# partial weeks at month boundaries, etc.
OFFSEASON_PATTERNS = [
    [0.22, 0.28, 0.28, 0.22],  # slight middle bump (common)
    [0.24, 0.26, 0.26, 0.24],  # nearly flat (stable period)
    [0.20, 0.30, 0.30, 0.20],  # stronger middle (minor outbreak)
    [0.23, 0.27, 0.27, 0.23],  # gentle slope up then down
]

# Outbreak months: strong mid-month spike (weeks 2-3)
# These get small random perturbation for year-to-year realism
OUTBREAK_BASE = [0.15, 0.35, 0.35, 0.15]

# True epidemiological week mapping (52-week year)
# Each month gets ~4.3 weeks; we distribute to avoid gaps

Month_nums = {
    "January":1, "February":2, "March":3, "April":4,
    "May":5, "June":6, "July":7, "August":8,
    "September":9, "October":10, "November":11, "December":12
}

MONTH_TO_START_WEEK = {
    1:1,  2:5,  3:9,  4:14,
    5:18, 6:22, 7:27, 8:31,
    9:36, 10:40, 11:44, 12:48
}

df=pd.read_csv('data/raw/dengue_data.csv')


In [ ]:
def allocate_exact(total, weights):
    """
    Distribute `total` across bins according to `weights`.
    Returns integer list that sums exactly to `total`.
    """
    raw = [total * w for w in weights]
    allocated = [int(x) for x in raw]
    remainder = total - sum(allocated)
    
    # Distribute remainder to largest fractional parts first (fairer)
    fractions = [(i, raw[i] - allocated[i]) for i in range(len(raw))]
    fractions.sort(key=lambda x: x[1], reverse=True)
    
    for i in range(remainder):
        idx = fractions[i % len(fractions)][0]
        allocated[idx] += 1
    
    return allocated

In [ ]:

weekly_rows = []

for _, row in df.iterrows():
    month = str(row["Month"])
    month_num = Month_nums[month]
    year = int(row["Year"])
    region = row["Region"]
    total_cases = int(row["Dengue_Cases"])
    total_deaths = int(row["Dengue_Deaths"])
    start_week = MONTH_TO_START_WEEK[month]
    
    # ── SELECT WEIGHTS ──
    if month_num in [7, 8, 9, 10]:  # Outbreak season
        
        rng = np.random.default_rng(
            hash(f"{year}{month_num}{region}") % (2**32)
        )
        noise = rng.normal(0, 0.02, 4)  # Small ±2% variation
        weights = [max(0.05, w + n) for w, n in zip(OUTBREAK_BASE, noise)]
    else:  # Off-season
        
        pattern_idx = (year + month_num + hash(region)) % len(OFFSEASON_PATTERNS)
        weights = list(OFFSEASON_PATTERNS[pattern_idx])
    
    # Normalize to ensure sum = 1.0
    weights = [w / sum(weights) for w in weights]
    
    # ── ALLOCATE CASES AND DEATHS ──
    week_cases = allocate_exact(total_cases, weights)
    week_deaths = allocate_exact(total_deaths, weights)
    
    # ── CREATE ROWS ──
    for w in range(4):
        epi_week = min(start_week + w, 52)
        weekly_rows.append({
            "year": year,
            "month": month,
            "month_num": month_num,
            "epi_week": epi_week,
            "region": region,
            "cases": week_cases[w],
            "deaths": week_deaths[w]
        })

df_weekly = pd.DataFrame(weekly_rows)


In [48]:
# verify totals are preserved per region per year
original_totals = df.groupby(["Year","Region"])["Dengue_Cases"].sum()
weekly_totals   = df_weekly.groupby(["year","region"])["cases"].sum()

print("Checking totals are preserved...")
for (year, region), orig in original_totals.items():

    weekly = weekly_totals.get((year, region), 0)

    diff = abs(orig - weekly)

    if diff > 4:  # allow rounding error of up to 4 cases
        print(f"  WARNING {year} {region}: original={orig} weekly_sum={weekly} diff={diff}")

print(f"\nWeekly rows generated: {len(df_weekly)}")
print(f"Original monthly rows: {len(df)}")
print(f"Expected weekly rows:  {len(df)} × 4 = {len(df)*4}")
print(f"\nSample — Punjab 2019 peak weeks:")
print(df_weekly[
    (df_weekly["region"]=="Punjab") &
    (df_weekly["year"]==2019) &
    (df_weekly["epi_week"].between(31,40))
].to_string())

df_weekly.to_csv("data/raw/pakistan_province_weekly.csv", index=False)
print("\nSaved pakistan_province_weekly.csv")

Checking totals are preserved...

Weekly rows generated: 1680
Original monthly rows: 420
Expected weekly rows:  420 × 4 = 1680

Sample — Punjab 2019 peak weeks:
      year      month  month_num  epi_week  region  cases  deaths
1036  2019     August          8        31  Punjab   1878       7
1037  2019     August          8        32  Punjab   4176      16
1038  2019     August          8        33  Punjab   4366      17
1039  2019     August          8        34  Punjab   2030       8
1040  2019  September          9        36  Punjab   1411       5
1041  2019  September          9        37  Punjab   2968      11
1042  2019  September          9        38  Punjab   3020      11
1043  2019  September          9        39  Punjab   1321       5
1044  2019    October         10        40  Punjab    509       2

Saved pakistan_province_weekly.csv
